# Einfache Lineare Regression

**Ziel:** Sie untersuchen den Zusammenhang zwischen **einer numerischen Eingabevariable** und **einer numerischen Zielvariable**, trainieren ein lineares Modell und bewerten dessen Vorhersagen.

**Für Advertising und Players sowie andere CSV-Dateien geeignet.** Sie müssen den Python-Code nicht vollständig verstehen: Ändern Sie nur die Werte in der **Konfigurationszelle** und führen Sie die Zellen nacheinander aus.

**Modell:** $\hat y = \hat\beta_0 + \hat\beta_1 x$

## 1. Datensatz hochladen
Laden Sie die CSV-Datei von Ihrem Computer hoch (z. B. `Advertising.csv` oder `players.csv`).

In [ ]:
#@title 1. CSV-Datei hochladen
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError('Bitte eine CSV-Datei auswählen.')

## 2. Daten ansehen
Prüfen Sie die vorhandenen Spaltennamen und die deskriptiven Statistiken.

In [ ]:
#@title 2. Daten einlesen und beschreiben
import io
import pandas as pd
import numpy as np

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
# Eventuelle exportierte Indexspalten entfernen
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed(:|$)')]

print('Datensatz:', filename)
print('Zeilen, Spalten:', df.shape)
print('\nSpalten:', list(df.columns))
print('\nNumerische Spalten:', df.select_dtypes(include='number').columns.tolist())
display(df.head())
display(df.describe(include='number').T.round(2))

## 3. Konfiguration – nur hier anpassen
Wählen Sie **genau eine** numerische Eingabevariable `X_VARIABLE` und die numerische Zielvariable `Y_VARIABLE`. Die Schreibweise muss zu den Spalten im Datensatz passen.

Advertising: `TV` → `sales`  ·  Players: z. B. `goalsTotal` → `marketvalue`.

In [ ]:
#@title 3. Variablen und Testanteil konfigurieren
# ===== KONFIGURATION: NUR HIER ANPASSEN =====
X_VARIABLE = 'TV'            # Players-Beispiel: 'goalsTotal'
Y_VARIABLE = 'sales'         # Players-Beispiel: 'marketvalue'
TEST_SIZE = 0.20             # 20 % Testdaten
RANDOM_STATE = 42            # gleiche zufällige Aufteilung
# ===========================================

if not isinstance(X_VARIABLE, str) or not isinstance(Y_VARIABLE, str):
    raise ValueError('Bitte für X_VARIABLE und Y_VARIABLE jeweils einen Spaltennamen als Text angeben.')
if X_VARIABLE == Y_VARIABLE:
    raise ValueError('Eingabevariable und Zielvariable müssen unterschiedlich sein.')
missing = [c for c in [X_VARIABLE, Y_VARIABLE] if c not in df.columns]
if missing:
    raise ValueError(f'Spalten nicht gefunden: {missing}. Vorhanden: {list(df.columns)}')
non_numeric = [c for c in [X_VARIABLE, Y_VARIABLE] if not pd.api.types.is_numeric_dtype(df[c])]
if non_numeric:
    raise ValueError(f'Nicht numerische Spalten: {non_numeric}.')
model_df = df[[X_VARIABLE, Y_VARIABLE]].replace([np.inf, -np.inf], np.nan).dropna().copy()
if len(model_df) < 10:
    raise ValueError('Zu wenige vollständige Beobachtungen (mindestens 10 erforderlich).')
print(f'Eingabevariable: {X_VARIABLE}  |  Zielvariable: {Y_VARIABLE}')
print(f'Verwendbare Beobachtungen: {len(model_df)} (entfernt: {len(df)-len(model_df)})')

## 4. Zusammenhang erkunden
Der Scatterplot zeigt, ob ein **annähernd linearer Zusammenhang** plausibel ist. Die Pearson-Korrelation misst nur den linearen Zusammenhang.

In [ ]:
#@title 4. Scatterplot und Korrelation
import matplotlib.pyplot as plt
print('Pearson-Korrelation:', round(model_df[X_VARIABLE].corr(model_df[Y_VARIABLE]), 3))
plt.figure(figsize=(7,4.5))
plt.scatter(model_df[X_VARIABLE], model_df[Y_VARIABLE], alpha=0.6)
plt.xlabel(X_VARIABLE)
plt.ylabel(Y_VARIABLE)
plt.title(f'{Y_VARIABLE} vs. {X_VARIABLE}')
plt.grid(alpha=0.2)
plt.show()

## 5. Trainings- und Testdaten bilden
Die Regression wird ausschließlich mit **Trainingsdaten** geschätzt. Die Testdaten dienen dazu, Vorhersagen auf unbekannten Beobachtungen zu bewerten.

In [ ]:
#@title 5. Trainings- und Testdaten aufteilen
from sklearn.model_selection import train_test_split
X = model_df[[X_VARIABLE]]   # doppelte Klammern: DataFrame für scikit-learn
y = model_df[Y_VARIABLE]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print('Training:', len(X_train), 'Beobachtungen')
print('Test:', len(X_test), 'Beobachtungen')

## 6. Lineare Regression trainieren
Der Algorithmus bestimmt Achsenabschnitt $\hat\beta_0$ und Steigung $\hat\beta_1$, indem er den **Trainings-RSS** minimiert.

In [ ]:
#@title 6. Modell trainieren und Funktion anzeigen
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
print(f'Achsenabschnitt (β₀): {model.intercept_:.5f}')
print(f'Steigung (β₁):        {model.coef_[0]:.5f}')
print(f'Gelernte Funktion: {Y_VARIABLE}_vorhergesagt = {model.intercept_:.5f} '
      f'{model.coef_[0]:+.5f} × {X_VARIABLE}')

## 7. Daten und Regressionsgerade darstellen
Die Gerade ist auf Basis der **Trainingsdaten** gelernt. Die Testpunkte wurden nicht zum Schätzen verwendet.

In [ ]:
#@title 7. Datenpunkte und geschätzte Gerade
import numpy as np
x_line = np.linspace(model_df[X_VARIABLE].min(), model_df[X_VARIABLE].max(), 200)
line_df = pd.DataFrame({X_VARIABLE: x_line})
y_line = model.predict(line_df)

plt.figure(figsize=(8,5))
plt.scatter(X_train[X_VARIABLE], y_train, alpha=0.55, label='Training')
plt.scatter(X_test[X_VARIABLE], y_test, alpha=0.7, label='Test')
plt.plot(x_line, y_line, color='black', linewidth=2, label='Regressionsgerade (Training)')
plt.xlabel(X_VARIABLE)
plt.ylabel(Y_VARIABLE)
plt.title('Einfache lineare Regression')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 8. Vorhersagen auf Testdaten und Modellfehler
**SSE/RSS:** Summe der quadrierten Fehler; **MSE:** mittlerer quadrierter Fehler; **RMSE:** Fehlermaß in der Einheit der Zielvariable; **MAE:** mittlerer absoluter Fehler; **R²:** Verbesserung gegenüber einer Mittelwertprognose auf denselben Testdaten.

In [ ]:
#@title 8. Testvorhersagen und Modellgüte
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y_pred = model.predict(X_test)
vergleich = X_test.copy()
vergleich['Istwert'] = y_test.to_numpy()
vergleich['Vorhersage'] = y_pred
vergleich['Residuum (Ist - Vorhersage)'] = vergleich['Istwert'] - vergleich['Vorhersage']
display(vergleich.head(10).round(3))

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
sse = float(np.sum((y_test.to_numpy() - y_pred) ** 2))
r2 = r2_score(y_test, y_pred)
print(f'R²      = {r2:.3f}')
print(f'MAE     = {mae:.3f}')
print(f'MSE     = {mse:.3f}')
print(f'RMSE    = {rmse:.3f}')
print(f'SSE/RSS = {sse:.3f}')
print('Alle Kennzahlen beziehen sich auf die Testdaten.')

## 9. Modellgüte visualisieren
Je näher die Punkte an der gestrichelten Diagonale liegen, desto genauer stimmen Vorhersage und Istwert überein.

In [ ]:
#@title 9. Istwerte und Vorhersagen vergleichen
plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred, alpha=0.7)
lo = min(float(y_test.min()), float(np.min(y_pred)))
hi = max(float(y_test.max()), float(np.max(y_pred)))
plt.plot([lo,hi], [lo,hi], 'k--', label='Perfekte Vorhersage')
plt.xlabel(f'Tatsächlicher Wert ({Y_VARIABLE})')
plt.ylabel(f'Vorhergesagter Wert ({Y_VARIABLE})')
plt.title('Testdaten: Istwert vs. Prognose')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 10. Reflexion

1. Ist die Wahl einer **linearen** Hypothese anhand des Scatterplots plausibel?
2. Wie lautet die gelernte Regressionsfunktion und wie interpretieren Sie die Steigung?
3. Wie hoch ist der **RMSE der Testdaten**? In welcher Einheit wird er gemessen?
4. Welche Testbeobachtungen haben besonders große Residuen?
5. Welche Zusammenhänge könnte das einfache lineare Modell **nicht** erfassen?

**Wichtig:** Bei großen Abweichungen ist ein linearer Zusammenhang nicht automatisch falsch: Andere, im Modell nicht berücksichtigte Einflussfaktoren können ebenfalls eine Rolle spielen.